# **🟢 CELL 1: Install Dependencies**

In [6]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio google-genai pydantic requests transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.7 MB/s eta 0:00:00:00:0100:01


# **🟢 CELL 2: Imports and Configurations**

In [7]:
import os
import shutil
import mimetypes
import tempfile
import threading
import requests
from contextlib import asynccontextmanager
from typing import List, Optional

import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
from google import genai
from google.genai import types
from kaggle_secrets import UserSecretsClient

# define user secrets
user_secrets = UserSecretsClient()

# Patch asyncio loop for Jupyter notebook environment
nest_asyncio.apply()

# Gemini Vision & API Configuration
api_keys = [user_secrets.get_secret("GEMINI_API_KEY")]
MODEL_FALLBACKS = ["gemini-2.5-flash", "gemini-1.5-flash"]

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_ID = "Mazenbassem/fiTpulse-fitness-model"
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

print("🚀 Loading base model into Kaggle GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

print("⚡ Applying your fine-tuned FitPulse weights...")
model = PeftModel.from_pretrained(base_model, MODEL_ID)

print("✅ FitPulse AI Model Ready!")

🚀 Loading base model into Kaggle GPU...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

⚡ Applying your fine-tuned FitPulse weights...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/162M [00:00<?, ?B/s]

✅ FitPulse AI Model Ready!


# **🟢 CELL 3: Helper Funtions(***Rag + Parser + Image Parser***)**

In [9]:
def parse_inbody_image(file_path: str) -> str:
    if not os.path.exists(file_path):
        return "Failed to parse file: File not found."

    with open(file_path, "rb") as f:
        file_bytes = f.read()

    # Dynamic MIME detection (PNG, JPEG, PDF)
    mime_type, _ = mimetypes.guess_type(file_path)
    if not mime_type:
        ext = os.path.splitext(file_path)[1].lower()
        if ext == ".pdf":
            mime_type = "application/pdf"
        elif ext == ".png":
            mime_type = "image/png"
        else:
            mime_type = "image/jpeg"

    prompt_text = """Analyze this scan/document sheet and extract key metrics:
1. Weight (kg)
2. Skeletal Muscle Mass / SMM (kg)
3. Body Fat Mass / BFM (kg)
4. Percent Body Fat / PBF (%)
5. Basal Metabolic Rate / BMR (kcal)
6. Muscle-Fat Graph Shape (C-shape, I-shape, or D-shape)
7. Segmental Lean Analysis (% of Standard)
8. Extra relevant details.

Return ONLY a concise bulleted list."""

    for key_idx, key in enumerate(api_keys, start=1):
        client = genai.Client(api_key=key)
        for model_name in MODEL_FALLBACKS:
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=[
                        types.Part.from_bytes(data=file_bytes, mime_type=mime_type),
                        prompt_text
                    ]
                )
                if response.text and response.text.strip():
                    return response.text.strip()
            except Exception as e:
                continue

    return "Failed to parse file after exhausting API options."


def process_inbody_and_consult(file_path: str, user_goal: str = "Recomposition") -> str:
    inbody_summary = parse_inbody_image(file_path)
    
    combined_prompt = f"""Here is my scan/report summary:
{inbody_summary}

User Goal: {user_goal}

Based on these numbers and my goal, provide:
1. Analysis of muscle-to-fat balance.
2. Tailored workout strategy.
3. Daily calories and protein targets based on BMR."""

    # return ask_fitpulse_with_memory(
    #     prompt=combined_prompt,
    #     use_rag=True
    # )
    return stream_fitpulse_with_memory(
        prompt=combined_prompt,
        use_rag=True
    )

In [19]:
from threading import Thread
from transformers import TextIteratorStreamer
import requests

FITPULSE_SYSTEM_PROMPT = (
    "You are FitPulse AI, an expert fitness, nutrition, and body composition coach. "
    "Format every reply in Markdown: use ## / ### headings to organize sections, "
    "**bold** for key numbers/terms, and - bullet or 1. numbered lists for steps, make sure h markdown elements can be rendered. "
    "Whenever you recommend a specific named exercise (not a general category), "
    "immediately follow its name with a tag on its own line in this exact format: "
    "[EXERCISE: {\"id\": \"lowercase_snake_case_id\", \"name\": \"Human Readable Exercise Name\"}] "
    "so the app can display a reference photo. Only tag exercises you are actively "
    "recommending, not passing mentions."
)

def stream_fitpulse_with_memory(
    prompt: str,
    history: Optional[List[dict]] = None,
    system_prompt: str = FITPULSE_SYSTEM_PROMPT,
    use_rag: bool = False
):
    """Streams token chunks directly from Kaggle GPU memory using PyTorch."""
    messages = [{"role": "system", "content": system_prompt}]
    if history:
        for msg in history:
            messages.append({"role": msg["role"], "content": msg["content"]})

    context_str = ""
    if use_rag and 'index' in globals():
        try:
            retriever = index.as_retriever(similarity_top_k=2)
            nodes = retriever.retrieve(prompt)
            if nodes:
                retrieved_chunks = [f"- {node.get_content()}" for node in nodes]
                context_str = "\n\nReference Knowledge:\n" + "\n".join(retrieved_chunks)
        except Exception as e:
            print(f"⚠️ RAG Retrieval warning: {e}")

    messages.append({"role": "user", "content": f"{prompt}{context_str}"})

    # Detect device dynamically (CUDA GPU or CPU fallback)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1. Format prompt with return_dict=True to get tensors for input_ids and attention_mask
    model_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(device)

    # 2. Set up real-time streaming streamer
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    
    # 3. Unpack model_inputs into generation_kwargs
    generation_kwargs = dict(
        **model_inputs,
        streamer=streamer,
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    # 4. Stream tokens using Dynamic Queue Draining for maximum network speed
    import queue

    while True:
        try:
            # Wait for the next token from the generator thread
            first_text = streamer.text_queue.get(timeout=15.0)
            if first_text is streamer.stop_signal:
                break

            chunk = first_text

            # Drain any extra tokens that accumulated in the queue while sending HTTP bytes
            while not streamer.text_queue.empty():
                next_text = streamer.text_queue.get_nowait()
                if next_text is streamer.stop_signal:
                    yield chunk
                    chunk = ""
                    break
                chunk += next_text

            if chunk:
                yield chunk

        except queue.Empty:
            break
    # Flush remaining text in buffer at the end
    if buffer:
        yield buffer
   

# **🟢 CELL 4: FASTAPI**

In [11]:
import base64
import tempfile
import os
from typing import List, Optional
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

# Safe handling of optional modules in lifespan
try:
    from pyngrok import ngrok
except ImportError:
    ngrok = None

try:
    import torch
except ImportError:
    torch = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    print("🚀 FitPulse API starting up...")
    yield
    print("🛑 Shutting down FitPulse API...")
    if ngrok:
        try:
            ngrok.kill()
        except Exception:
            pass
    if torch and torch.cuda.is_available():
        torch.cuda.empty_cache()


app = FastAPI(title="FitPulse AI Backend API", lifespan=lifespan)

# Allow cross-origin requests from frontend / ngrok dashboard
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    query: str
    inbody_image: Optional[str] = None
    history: Optional[List[ChatMessage]] = []
    use_rag: Optional[bool] = True


@app.get("/")
def health_check():
    return {"status": "online", "model": "FitPulse AI (Kaggle Backend)"}


def to_stream_generator(result):
    """
    Flexible streamer: Yields tokens immediately if your notebook function returns a 
    generator/streamer, or safely chunks the output if it returns a single string.
    """
    if isinstance(result, str):
        chunk_size = 32
        for i in range(0, len(result), chunk_size):
            yield result[i : i + chunk_size]
    else:
        # Handles generators, TextIteratorStreamer, or chunk lists
        for chunk in result:
            if isinstance(chunk, str):
                yield chunk
            elif hasattr(chunk, "text"):
                yield chunk.text
            elif isinstance(chunk, dict) and "content" in chunk:
                yield chunk["content"]


STREAM_HEADERS = {
    "X-Accel-Buffering": "no",
    "Cache-Control": "no-cache",
    "Connection": "keep-alive",
}

@app.post("/api/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        formatted_history = (
            [{"role": msg.role, "content": msg.content} for msg in req.history]
            if req.history else []
        )

        # Case 1: InBody scan image uploaded — restored regression.
        # Vision analysis isn't token-streamable, so we generate the full
        # reply then chunk it through to_stream_generator (already defined
        # above, was just orphaned).
        if req.inbody_image and req.inbody_image.strip():
            img_data = req.inbody_image.strip()
            if "," in img_data:
                img_data = img_data.split(",", 1)[1]

            temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
            temp_file.write(base64.b64decode(img_data))
            temp_file.close()
            temp_path = temp_file.name

            def stream_inbody_and_cleanup(path: str, goal: str):
                try:
                    reply = process_inbody_and_consult(file_path=path, user_goal=goal)
                    for chunk in to_stream_generator(reply):
                        yield chunk
                except Exception as e:
                    yield f"\n[InBody analysis error: {str(e)}]"
                finally:
                    if os.path.exists(path):
                        os.remove(path)

            return StreamingResponse(
                stream_inbody_and_cleanup(temp_path, req.query),
                media_type="text/plain",
                headers=STREAM_HEADERS,
            )

        # Case 2: regular text chat — real token-by-token stream from the local model
        return StreamingResponse(
            stream_fitpulse_with_memory(
                prompt=req.query,        # fixed: was req.prompt (field doesn't exist on ChatRequest)
                history=formatted_history,
                use_rag=req.use_rag
            ),
            media_type="text/plain",
            headers=STREAM_HEADERS,
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# **🟢 CELL 5: NGROK Tunnel**

In [12]:
import requests

# Paste your Cloudflare URL here:
#CLOUDFLARE_URL = "https://lobby-poor-wing-provided.trycloudflare.com"

try:
    response = requests.get(OLLAMA_BASE_URL)
    print("✅ SUCCESS FROM KAGGLE:", response.text)
except Exception as e:
    print("❌ FAILED TO REACH TUNNEL:", e)

❌ FAILED TO REACH TUNNEL: name 'OLLAMA_BASE_URL' is not defined


In [ ]:
from pyngrok import ngrok
import uvicorn
import threading
import time
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Set your Ngrok Auth Token
NGROK_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_TOKEN)


# 1. Close active pyngrok tunnel instances
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    ngrok.kill()
    print("🧹 Cleared active pyngrok sessions.")
except Exception as e:
    print(f"Info: {e}")

# 2. Hard-kill any lingering ngrok background processes in Linux
os.system("pkill -9 -f ngrok")
time.sleep(1) # Small pause for socket cleanup
print("⚡ Killed lingering ngrok system processes.")


# Reset Ngrok tunnels
try:
    ngrok.kill()
except Exception:
    pass

# Open Ngrok tunnel on port 8000
public_url = ngrok.connect(8000, pooling_enabled=True )
print(f"\n🚀 REACT ENDPOINT URL:\n{public_url.public_url}/api/chat\n")

# Pass `app` directly as the Python object from Cell 6
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

# Run Uvicorn in top-level await (or via background thread)
await server.serve()

🧹 Cleared active pyngrok sessions.
⚡ Killed lingering ngrok system processes.

🚀 REACT ENDPOINT URL:
https://gumdrop-paralyses-replica.ngrok-free.dev/api/chat



INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 FitPulse API starting up...
INFO:     154.180.239.57:0 - "POST /api/chat HTTP/1.1" 200 OK


ERROR:    Exception in ASGI application
  + Exception Group Traceback (most recent call last):
  |   File "/usr/local/lib/python3.12/dist-packages/starlette/_utils.py", line 81, in collapse_excgroups
  |     yield
  |   File "/usr/local/lib/python3.12/dist-packages/starlette/responses.py", line 270, in __call__
  |     async with anyio.create_task_group() as task_group:
  |                ^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/usr/local/lib/python3.12/dist-packages/anyio/_backends/_asyncio.py", line 799, in __aexit__
  |     raise BaseExceptionGroup(
  | ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception)
  +-+---------------- 1 ----------------
    | Traceback (most recent call last):
    |   File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    |     result = await app(  # type: ignore[func-returns-value]
    |              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    |   File "/usr/local/lib/python3.12/